> ⚠️ **실행 안내 및 데이터 공지**
>
> 1. **데이터 보안 및 용량:** 원본(raw) 데이터는 보안 및 대용량 문제로 저장소에 포함되지 않았습니다.
> 2. **참고용 코드:** 본 노트북(01~02)은 전처리 로직을 설명하기 위한 참고용입니다.
> 3. **실행 가능 지점:** 분석 재현 및 모델 실행은 **`03_data_preparation.ipynb`부터 가능**합니다. (`data/processed/` 사용)

# 02. 데이터 전처리 및 파생변수 생성

**분석 목적:** 원본 데이터를 분석 가능한 형태로 정제하고,
수익 분석에 필요한 파생변수를 생성한다

**핵심 작업:**
- 결측치 처리 및 district 복원
- 불필요 변수 제거
- 숙소 상태 및 그룹 변수 생성
- 이상치 제거
- 외부 데이터(관광지, 생활인구) 병합

## 0. 환경 설정

In [ ]:
# OS별 한글 폰트 자동 설정 -- Mac/Windows/Linux 모두 대응
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False
import ssl
import warnings

# 한글 폰트 설정
pd.options.display.float_format = '{:,.2f}'.format
warnings.filterwarnings('ignore')
ssl._create_default_https_context = ssl._create_unverified_context

## 1. 데이터 로딩

In [ ]:
# district 컬럼에 1,198건 NaN 존재 -> 다음 단계에서 Spatial Join으로 복원
df = pd.read_csv('../data/raw/seoul_airbnb_geo_masked.csv')

print(f'\n결측치 현황:')
print(df.isnull().sum()[df.isnull().sum() > 0])

---
## 2. 결측치 처리: district NaN → Spatial Join 

- district 컬럼에 1,198건의 NaN 존재
- 서울시 행정구역 GeoJSON과 숙소 위경도를 Spatial Join하여 결측치 보완
- 좌표가 서울 경계 밖인 경우에도 가장 가까운 구에 매핑

In [ ]:
# district NaN을 좌표 기반 Spatial Join으로 복원 -- 최빈값 대체 시 구 정보 오염
import geopandas as gpd
import requests
from shapely.geometry import Point

print(f'[Before] district 결측치: {df["district"].isnull().sum():,}건')

# 서울시 행정구역 GeoJSON 로드
geojson_url = 'https://raw.githubusercontent.com/southkorea/seoul-maps/master/kostat/2013/json/seoul_municipalities_geo_simple.json'
response = requests.get(geojson_url, verify=False)
geojson_data = response.json()

seoul_gdf = gpd.GeoDataFrame.from_features(geojson_data['features'])
seoul_gdf.set_crs(epsg=4326, inplace=True)

# 숙소 좌표를 GeoDataFrame으로 변환
geometry = [Point(xy) for xy in zip(df['longitude_masked'], df['latitude_masked'])]
airbnb_gdf = gpd.GeoDataFrame(df, geometry=geometry)
airbnb_gdf.set_crs(epsg=4326, inplace=True)

# Spatial Join: 각 숙소가 어떤 구 안에 있는지 매핑
joined = gpd.sjoin(airbnb_gdf, seoul_gdf[['name', 'geometry']], how='left', predicate='within')

# 한글 → 영문 구 이름 매핑
district_map = {
    '강남구': 'Gangnam-gu', '강동구': 'Gangdong-gu', '강북구': 'Gangbuk-gu',
    '강서구': 'Gangseo-gu', '관악구': 'Gwanak-gu', '광진구': 'Gwangjin-gu',
    '구로구': 'Guro-gu', '금천구': 'Geumcheon-gu', '노원구': 'Nowon-gu',
    '도봉구': 'Dobong-gu', '동대문구': 'Dongdaemun-gu', '동작구': 'Dongjak-gu',
    '마포구': 'Mapo-gu', '서대문구': 'Seodaemun-gu', '서초구': 'Seocho-gu',
    '성동구': 'Seongdong-gu', '성북구': 'Seongbuk-gu', '송파구': 'Songpa-gu',
    '양천구': 'Yangcheon-gu', '영등포구': 'Yeongdeungpo-gu', '용산구': 'Yongsan-gu',
    '은평구': 'Eunpyeong-gu', '종로구': 'Jongno-gu', '중구': 'Jung-gu',
    '중랑구': 'Jungnang-gu'
}

joined['mapped_district'] = joined['name'].map(district_map)
df['district'] = df['district'].fillna(joined['mapped_district'])

print(f'[After]  district 결측치: {df["district"].isnull().sum():,}건')

---
## 3. 불필요 컬럼 제거

| 컬럼 | 제거 사유 |
|------|----------|
| `cleaning_fee` | 93%가 0원, 분석 변별력 없음 |
| `country` | 모든 값이 'South Korea' (단일값) |
| `currency` | 모든 값이 'KRW' (단일값) |
| `locality` | 모든 값이 'Seoul' (단일값) |
| `region` | 모든 값이 'Seoul' (단일값) |

In [ ]:
# 단일값 컬럼(country/currency 등)은 정보 없음 -> 제거, cleaning_fee는 93%가 0원
drop_cols = ['cleaning_fee', 'country', 'currency', 'locality', 'region']


df = df.drop(columns=drop_cols)

---
## 4. 파생변수 생성

### 4-1. refined_status: 숙소 활동 상태 분류 

리뷰 유무와 점유율을 기준으로 숙소를 5개 그룹으로 분류

| 상태 | 조건 |
|------|------|
| `Active` | 리뷰 있음 + 최근 90일 점유율 > 0 |
| `New` | 리뷰 없음 + 최근 90일 점유율 > 0 |
| `Dormant` | 리뷰 있음 + 최근 90일 점유율 = 0 |
| `Mystery` | 리뷰 없음 + 최근 90일 점유율 = 0 + TTM 점유율 > 0 |
| `Ghost` | 리뷰 없음 + 최근 90일 점유율 = 0 + TTM 점유율 = 0 |

In [ ]:
# 리뷰/점유율 기반 5단계 분류 -- 단순 등록 여부가 아닌 실제 운영 실태를 반영
def refined_classify(row):
    has_review = row['num_reviews'] > 0
    recent_active = row['l90d_occupancy'] > 0
    past_active = row['ttm_occupancy'] > 0

    if has_review and recent_active:
        return 'Active'
    elif not has_review and recent_active:
        return 'New'
    elif has_review and not recent_active:
        return 'Dormant'
    elif not has_review and not recent_active and past_active:
        return 'Mystery'
    else:
        return 'Ghost'

df['refined_status'] = df.apply(refined_classify, axis=1)

print('숙소 활동 상태 분류 (refined_status):')
print(df['refined_status'].value_counts())
print(f'\n비율:')
print(df['refined_status'].value_counts(normalize=True).round(4))

### 4-2. baths_group: 욕실 수 그룹 

- 욕실 5개 기준 2그룹 분류
- `Normal`: 5개 미만 / `Large`: 5개 이상

In [ ]:
# 욕실 수를 Normal/Large로 이진화 -- 모델 피처 복잡도 축소 목적
df['baths_group'] = np.where(df['baths'] >= 5, 'Large', 'Normal')

baths_counts = df['baths_group'].value_counts()
print('욕실(baths) — 2그룹 분류 (baths_group):')
print(f'  [Normal] 욕실 5개 미만: {baths_counts.get("Normal", 0):,}건')
print(f'  [Large]  욕실 5개 이상: {baths_counts.get("Large", 0):,}건')

### 4-3. bedrooms_group: 침실 수 그룹 

- 침실 5개 기준 2그룹 분류
- `Normal`: 5개 미만 / `Large`: 5개 이상

In [ ]:
# 침실 수 이진화 (5개 기준) -- 대형 숙소와 일반 숙소를 구분하는 핵심 기준
df['bedrooms_group'] = np.where(df['bedrooms'] >= 5, 'Large', 'Normal')

bedrooms_counts = df['bedrooms_group'].value_counts()
print('침실(bedrooms) — 2그룹 분류 (bedrooms_group):')
print(f'  [Normal] 침실 5개 미만: {bedrooms_counts.get("Normal", 0):,}건')
print(f'  [Large]  침실 5개 이상: {bedrooms_counts.get("Large", 0):,}건')

### 4-4. guests_group: 게스트 수용 인원 그룹 

- 게스트 수 7명 기준 2그룹 분류
- `소규모 그릅`: 0~6명 (guests=0은 수용 인원 미기재) / `대규모 그룹`: 7명 이상 (대형/단체 숙소)

In [ ]:
# 수용 인원 이진화 (7명 기준) -- 단체 숙소 여부 구분
df['guests_group'] = np.where(df['guests'] >= 7, '7+', '1_6')

guests_counts = df['guests_group'].value_counts()
print('게스트(guests) — 2그룹 분류 (guests_group):')
print(f" [소규모 그룹] (0~6명): {guests_counts.get('1_6', 0):,}건")
print(f" [대규모 그룹]  (7명+): {guests_counts.get('7+', 0):,}건")

### 4-5. extra_guest_fee_policy: 추가 게스트 요금 유무 플래그 

- 추가 게스트 요금(extra_guest_fee) > 0이면 1, 아니면 0
- 전체의 93.2%가 0원 (요금 정책 없음) → 참고용 변수

In [ ]:
# 93%가 0원 -> 이진 플래그화, 모델에서 정책 유무의 효과만 측정
df['extra_guest_fee_policy'] = np.where(df['extra_guest_fee'] > 0, 1, 0)

fee_counts = df['extra_guest_fee_policy'].value_counts()
print('추가 게스트 요금 — 참고용 플래그 (extra_guest_fee_policy):')
print(f'  [0] 정책 없음 (0원): {fee_counts.get(0, 0):,}건')
print(f'  [1] 정책 있음 (1원+): {fee_counts.get(1, 0):,}건')

### 4-6. operation_status: 운영/비운영 숙소 분류 

- 아래 8개 수익/가동 지표가 **모두 0**이면 비운영(Non_Operating)으로 분류
- 대상 지표: `l90d_avg_rate`, `l90d_occupancy`, `l90d_revenue`, `l90d_revpar`, `ttm_avg_rate`, `ttm_occupancy`, `ttm_revenue`, `ttm_revpar`

In [ ]:
# 수익/가동 지표 8개 모두 0 -> 실제 미운영 숙소 분류 -- RevPAR 왜곡 원인 제거
operation_cols = [
    'l90d_avg_rate', 'l90d_occupancy', 'l90d_revenue', 'l90d_revpar',
    'ttm_avg_rate', 'ttm_occupancy', 'ttm_revenue', 'ttm_revpar'
]

all_zero_mask = (df[operation_cols] == 0).all(axis=1)
df['operation_status'] = np.where(all_zero_mask, 'Non_Operating', 'Operating')

op_counts = df['operation_status'].value_counts()
print('운영/비운영 숙소 분류 (operation_status):')
print(f'  [Operating]     실제 운영 숙소: {op_counts.get("Operating", 0):,}건')
print(f'  [Non_Operating] 비운영 숙소:   {op_counts.get("Non_Operating", 0):,}건')

print('\n비운영 숙소 특성 요약:')
non_op = df[df['operation_status'] == 'Non_Operating']
print(f'  - 평균 리뷰 수: {non_op["num_reviews"].mean():.1f}건')
print(f'  - 리뷰 0개 비율: {(non_op["num_reviews"] == 0).mean():.1%}')
print(f'  - 평균 평점: {non_op["rating_overall"].mean():.2f}')

---
## 5. 이상치 제거

### 5-1. min_nights > 730 제거 

- 에어비앤비 시스템상 최소 숙박일은 최대 730일(2년)까지 설정 가능
- 730일 초과 데이터 16건은 시스템 오류로 판단하여 제거

In [ ]:
# 에어비앤비 최대 설정값(730일) 초과 -> 시스템 입력 오류로 판단해 제거
min_nights_outlier = df[df['min_nights'] > 730]

print(f'[Before] 데이터: {len(df):,}건 | min_nights 최대값: {df["min_nights"].max():,.0f}')
print(f'  제거 대상: {len(min_nights_outlier):,}건 (min_nights > 730)')

df = df.drop(min_nights_outlier.index)

print(f'[After]  데이터: {len(df):,}건 | min_nights 최대값: {df["min_nights"].max():,.0f}')

### 5-2. ttm_avg_rate > 2,000,000 제거

- 1박 평균 요금 200만원 초과 숙소 24건 중 23건은 점유율 0% (허수 데이터)
- 실제 120~160만원대 숙소는 정상 운영 확인되어 유지
- IQR 대신 도메인 기준(200만원)으로 이상치 판단

In [ ]:
# 200만원 초과 24건 중 23건이 점유율 0% -> IQR 대신 도메인 기준으로 이상치 판단
HARD_LIMIT = 2_000_000
ttm_outlier = df[df['ttm_avg_rate'] > HARD_LIMIT]

print(f'[Before] 데이터: {len(df):,}건 | ttm_avg_rate 최대값: {df["ttm_avg_rate"].max():,.0f}')
print(f'  제거 대상: {len(ttm_outlier):,}건 (ttm_avg_rate > {HARD_LIMIT:,})')

df = df.drop(ttm_outlier.index)

print(f'[After]  데이터: {len(df):,}건 | ttm_avg_rate 최대값: {df["ttm_avg_rate"].max():,.0f}')

---
## 6. 최종 데이터 확인

In [ ]:
print(f'결측치 확인:')
missing_final = df.isnull().sum()
if missing_final.sum() > 0:
    print(missing_final[missing_final > 0])
else:
    print('  없음')

## 7. 전처리 완료 데이터 저장

In [ ]:
output_path = '../data/interim/seoul_airbnb_cleaned.csv'

df.to_csv(output_path, index=False)
print(f'전처리 완료 데이터 저장: {output_path}')

---
## 8. 외부 데이터 병합: 관광지 POI (seoul_tourism_all.csv)

각 숙소에 **가장 가까운 관광지**의 정보를 병합합니다.

**병합 방식:**
- 숙소 위경도(`latitude_masked`, `longitude_masked`)와 관광지 위경도(`위도`, `경도`)를 이용
- 각 숙소에서 직선 거리가 가장 가까운 관광지 1건을 매칭 (BallTree 최근접 이웃 탐색)
- 매칭된 관광지의 컬럼을 숙소 데이터에 추가

**추가되는 컬럼 (7개):**

| 컬럼 | 설명 |
|------|------|
| `nearest_poi_name` | 가장 가까운 관광지명 |
| `nearest_poi_addr` | 관광지 주소 |
| `nearest_poi_type` | 관광지 유형 코드 |
| `nearest_poi_type_name` | 관광지 유형명 (관광지/문화시설/음식점/쇼핑 등) |
| `nearest_poi_lat` | 관광지 위도 |
| `nearest_poi_lng` | 관광지 경도 |
| `nearest_poi_image` | 관광지 대표 이미지 URL |

### 8-1. 관광지 데이터 로딩

In [ ]:
# 한국관광공사 API 데이터 -- 유형별 건수로 데이터 품질 사전 확인
tourism = pd.read_csv('../data/external/seoul_tourism_all.csv')

print(f'관광 유형별 건수:')
print(tourism['타입'].value_counts().to_string())

### 8-2. 최근접 관광지 매칭 (BallTree)

각 숙소에서 가장 가까운 관광지 1건을 찾아 해당 관광지의 7개 속성을 매핑합니다.

In [ ]:
# BallTree(Haversine) 사용 -- 구면 거리 계산으로 위경도 왜곡 방지, 전체 루프보다 빠름
from sklearn.neighbors import BallTree

tourism_valid = tourism.dropna(subset=['위도', '경도']).reset_index(drop=True)
print(f'유효 관광지: {len(tourism_valid):,}건 (위경도 결측 제거)')

tourism_type_map = {
    12: '관광지',
    14: '문화시설',
    15: '축제공연행사',
    25: '여행코스',
    28: '레포츠',
    32: '숙박',
    38: '쇼핑',
    39: '음식점'
}
tourism_valid['tourism_type_name'] = tourism_valid['타입'].map(tourism_type_map)

poi_coords = np.radians(tourism_valid[['위도', '경도']].values)
tree = BallTree(poi_coords, metric='haversine')

airbnb_coords = np.radians(df[['latitude_masked', 'longitude_masked']].values)
dist, idx = tree.query(airbnb_coords, k=1)

dist_km = dist.flatten() * 6371

nearest = tourism_valid.iloc[idx.flatten()]

df['nearest_poi_name'] = nearest['관광지명'].values
df['nearest_poi_addr'] = nearest['주소'].values
df['nearest_poi_type'] = nearest['타입'].values
df['nearest_poi_type_name'] = nearest['tourism_type_name'].values
df['nearest_poi_lat'] = nearest['위도'].values
df['nearest_poi_lng'] = nearest['경도'].values
df['nearest_poi_image'] = nearest['firstimage'].values
df['nearest_poi_dist_km'] = dist_km.round(3)

print(f'최근접 거리 통계 (km):')
print(df['nearest_poi_dist_km'].describe().round(3))

### 8-3. 병합 결과 확인

In [ ]:
# 매칭 결과 검증 -- POI 컬럼 결측치 없음을 확인
poi_columns = ['nearest_poi_name', 'nearest_poi_addr', 'nearest_poi_type',
               'nearest_poi_type_name', 'nearest_poi_lat', 'nearest_poi_lng',
               'nearest_poi_image', 'nearest_poi_dist_km']

poi_missing = df[poi_columns].isnull().sum()
if poi_missing.sum() > 0:
    print('POI 컬럼 결측치:', poi_missing[poi_missing > 0].to_dict())
print(f'매칭된 관광지 유형 Top5:')
print(df['nearest_poi_type_name'].value_counts().head(5).to_string())

## 10. 외부 데이터 병합: 생활인구 (monthly_seoul_pop_final.csv)

**목표:** 시군구별 생활인구를 두 가지 기간으로 집계하여 숙소 데이터에 병합

| 파생변수 | 기간 | 설명 |
|----------|------|------|
| `l90_pop` | 2025-07 ~ 2025-09 (3개월 평균) | 최근 90일 생활인구 |
| `ttm_pop` | 2024-10 ~ 2025-09 (12개월 평균) | 연간 생활인구 |

**병합 방식:** `district` (영문) ↔ `시군구명` (한글) 매핑 딕셔너리 → Left Join


In [ ]:
# 서울시 합산 행 제거 후 구 단위 집계 -- TTM(12개월)과 L90(3개월) 두 기간으로 구분
pop = pd.read_csv('../data/external/monthly_population.csv')

pop = pop[pop['시군구명'] != '서울시'].copy()
l90_months = ['2025-07', '2025-08', '2025-09']
l90_pop = (pop[pop['기준년월'].isin(l90_months)]
           .groupby('시군구명')['총생활인구수']
           .mean()
           .astype(int)
           .reset_index())
l90_pop.columns = ['시군구명', 'l90_pop']

# 3️) ttm_pop: 최근 12개월 (2024-10 ~ 2025-09) 총생활인구수 평균
ttm_pop = (pop.groupby('시군구명')['총생활인구수']
           .mean()
           .astype(int)
           .reset_index())
ttm_pop.columns = ['시군구명', 'ttm_pop']

# 4️) 두 집계 결과 합치기
pop_summary = l90_pop.merge(ttm_pop, on='시군구명')

print(f'\n시군구별 생활인구 요약:')

In [ ]:
# 한글 시군구명 -> 영문 district 키로 변환 -- 숙소 데이터와 조인 키 통일
district_map_pop = {
    '강남구': 'Gangnam-gu', '강동구': 'Gangdong-gu', '강북구': 'Gangbuk-gu',
    '강서구': 'Gangseo-gu', '관악구': 'Gwanak-gu', '광진구': 'Gwangjin-gu',
    '구로구': 'Guro-gu', '금천구': 'Geumcheon-gu', '노원구': 'Nowon-gu',
    '도봉구': 'Dobong-gu', '동대문구': 'Dongdaemun-gu', '동작구': 'Dongjak-gu',
    '마포구': 'Mapo-gu', '서대문구': 'Seodaemun-gu', '서초구': 'Seocho-gu',
    '성동구': 'Seongdong-gu', '성북구': 'Seongbuk-gu', '송파구': 'Songpa-gu',
    '양천구': 'Yangcheon-gu', '영등포구': 'Yeongdeungpo-gu', '용산구': 'Yongsan-gu',
    '은평구': 'Eunpyeong-gu', '종로구': 'Jongno-gu', '중구': 'Jung-gu',
    '중랑구': 'Jungnang-gu'
}

pop_summary['district'] = pop_summary['시군구명'].map(district_map_pop)

unmapped = pop_summary[pop_summary['district'].isna()]['시군구명'].unique()
if len(unmapped) > 0:
    print(f' 매핑 실패: {unmapped}')
else:
    print(' 모든 시군구명 → district 매핑 완료')

pop_summary = pop_summary.drop(columns=['시군구명'])
print(f'\n병합용 인구 데이터:')
print(pop_summary.to_string(index=False))

In [ ]:
# 생활인구를 Left Join 후 RevPAR 관련 컬럼 바로 뒤에 삽입 -- 가독성 향상

df = df.merge(pop_summary, on='district', how='left')


# 7️) 컬럼 순서 조정: l90_pop → l90d_revpar 뒤, ttm_pop → ttm_revpar 뒤
cols = df.columns.tolist()
if 'l90_pop' in cols:
    cols.remove('l90_pop')
if 'ttm_pop' in cols:
    cols.remove('ttm_pop')

l90d_idx = cols.index('l90d_revpar') + 1
cols.insert(l90d_idx, 'l90_pop')

ttm_idx = cols.index('ttm_revpar') + 1
cols.insert(ttm_idx, 'ttm_pop')

df = df[cols]

print(f'\n병합 후 l90_pop 결측: {df["l90_pop"].isnull().sum()}건')
print(f'병합 후 ttm_pop 결측: {df["ttm_pop"].isnull().sum()}건')

print(f'\n병합 결과 샘플 (상위 5건):')
df[['district', 'l90d_revpar', 'l90_pop','ttm_revpar', 'ttm_pop']].head()

## 11. 관광지 근접도 파생변수: nearest_500m, nearest_1km

각 숙소 좌표 기준으로 **반경 내 관광지 개수**를 집계하여 2개의 파생변수를 생성합니다.

| 컬럼 | 설명 |
|------|------|
| `nearest_500m` | 숙소에서 반경 500m 이내 관광지 수 |
| `nearest_1km` | 숙소에서 반경 1km 이내 관광지 수 |

**산출 방식:**
- 숙소 위경도(`latitude_masked`, `longitude_masked`)와 관광지 위경도(`위도`, `경도`)를 이용
- BallTree(Haversine) 반경 탐색(`query_radius`)으로 각 숙소 주변 관광지 수 집계

In [ ]:
# 단순 최근접 POI 거리 외에 반경 500m/1km 내 POI 밀도 추가 -- 관광 집중도 반영
from sklearn.neighbors import BallTree

tourism = pd.read_csv('../data/external/seoul_tourism_all.csv')

tourism_valid = tourism.dropna(subset=['위도', '경도']).reset_index(drop=True)
poi_coords = np.radians(tourism_valid[['위도', '경도']].values)
tree = BallTree(poi_coords, metric='haversine')

airbnb_coords = np.radians(df[['latitude_masked', 'longitude_masked']].values)

EARTH_RADIUS_KM = 6371
radius_500m = 0.5 / EARTH_RADIUS_KM   # 500m
radius_1km = 1.0 / EARTH_RADIUS_KM    # 1km

counts_500m = tree.query_radius(airbnb_coords, r=radius_500m, count_only=True)
counts_1km = tree.query_radius(airbnb_coords, r=radius_1km, count_only=True)

df['nearest_500m'] = counts_500m
df['nearest_1km'] = counts_1km

print(df['nearest_500m'].describe().round(2))
print(df['nearest_1km'].describe().round(2))

In [ ]:
output_path = '../data/interim/final_seoul_airbnb_cleaned.csv'
df.to_csv(output_path, index=False)

print(f'최종 데이터 저장: {output_path}')
print(f'\n추가된 컬럼: nearest_500m, nearest_1km')
print(f'\n전체 컬럼 목록:')